# Auditoria da Base dos Dados

Este notebook verifica a viabilidade de uso da Base dos Dados como fonte auxiliar para inventario, validacao cruzada e enriquecimento das analises sobre sifilis congenita em Porto Alegre.

A auditoria nao assume uma janela temporal fixa. O periodo disponivel deve ser descoberto por consulta em cada tabela.

## Pergunta analitica

Quais tabelas da Base dos Dados possuem cobertura, granularidade e custo estimado adequados para complementar ou validar as bases DATASUS usadas no projeto?

## Bibliotecas e configuracao

In [ ]:
from __future__ import annotations

import os
from dataclasses import dataclass
from pathlib import Path

import pandas as pd

try:
    import basedosdados as bd
except ImportError:
    bd = None

try:
    from google.cloud import bigquery
except ImportError:
    bigquery = None

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 100)

GOOGLE_CLOUD_PROJECT = os.getenv("GOOGLE_CLOUD_PROJECT") or os.getenv("GCP_PROJECT") or ""
MAX_BYTES_BILLED = int(os.getenv("BIGQUERY_MAX_BYTES_BILLED", str(2 * 1024**3)))
EXECUTAR_CONSULTAS = os.getenv("EXECUTAR_CONSULTAS_BIGQUERY", "0") == "1"

print("basedosdados:", "disponivel" if bd is not None else "nao instalado")
print("google-cloud-bigquery:", "disponivel" if bigquery is not None else "nao instalado")
print("GOOGLE_CLOUD_PROJECT:", GOOGLE_CLOUD_PROJECT or "nao configurado")
print("EXECUTAR_CONSULTAS:", EXECUTAR_CONSULTAS)

## Tabelas candidatas

In [ ]:
@dataclass(frozen=True)
class TabelaCandidata:
    tabela: str
    uso_previsto: str
    decisao_inicial: str

    @property
    def dataset(self) -> str:
        return self.tabela.split(".")[1]

    @property
    def nome_tabela(self) -> str:
        return self.tabela.split(".")[2]


TABELAS = [
    TabelaCandidata(
        "basedosdados.br_ms_sinasc.microdados",
        "Nascidos vivos, denominador de incidencia e perfil materno",
        "validar",
    ),
    TabelaCandidata(
        "basedosdados.br_ms_sim.microdados",
        "Mortalidade e desfechos agregados",
        "validar",
    ),
    TabelaCandidata(
        "basedosdados.br_ms_cnes.estabelecimento",
        "Oferta assistencial por municipio/ano",
        "validar",
    ),
    TabelaCandidata(
        "basedosdados.br_ms_populacao.municipio",
        "Populacao municipal e contexto demografico",
        "validar",
    ),
    TabelaCandidata(
        "basedosdados.br_ms_sih.servicos_profissionais",
        "Contexto hospitalar/assistencial complementar",
        "opcional",
    ),
    TabelaCandidata(
        "basedosdados.br_ms_sinan.microdados_violencia",
        "Referencia tecnica para padrao de consumo SINAN, nao para sifilis congenita",
        "referencia",
    ),
]

pd.DataFrame([t.__dict__ for t in TABELAS])

## Funcoes de auditoria

In [ ]:
CANDIDATOS_ANO = ["ano", "ano_obito", "ano_nascimento", "ano_competencia", "ano_referencia"]
CANDIDATOS_UF = ["sigla_uf", "sigla_uf_residencia", "sigla_uf_estabelecimento"]
CANDIDATOS_MUNICIPIO = [
    "id_municipio",
    "id_municipio_residencia",
    "id_municipio_estabelecimento",
    "id_municipio_nascimento",
]
CODIGO_PORTO_ALEGRE_7 = "4314902"


def obter_cliente_bigquery():
    if bigquery is None or not GOOGLE_CLOUD_PROJECT:
        return None
    return bigquery.Client(project=GOOGLE_CLOUD_PROJECT)


def executar_sql(client, sql: str) -> pd.DataFrame:
    job_config = bigquery.QueryJobConfig(maximum_bytes_billed=MAX_BYTES_BILLED)
    rows = client.query(sql, job_config=job_config).result()
    return pd.DataFrame([dict(row.items()) for row in rows])


def estimar_bytes(client, sql: str) -> int | None:
    if client is None:
        return None
    job_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False, maximum_bytes_billed=MAX_BYTES_BILLED)
    job = client.query(sql, job_config=job_config)
    return int(job.total_bytes_processed or 0)


def consulta_colunas(tabela: TabelaCandidata) -> str:
    return f"""
    SELECT column_name, data_type
    FROM `basedosdados.{tabela.dataset}.INFORMATION_SCHEMA.COLUMNS`
    WHERE table_name = '{tabela.nome_tabela}'
    ORDER BY ordinal_position
    """


def primeira_coluna_existente(colunas: set[str], candidatas: list[str]) -> str | None:
    for coluna in candidatas:
        if coluna in colunas:
            return coluna
    return None


def montar_consulta_periodo(tabela: TabelaCandidata, colunas: set[str]) -> tuple[str, str | None, str | None]:
    coluna_ano = primeira_coluna_existente(colunas, CANDIDATOS_ANO)
    coluna_uf = primeira_coluna_existente(colunas, CANDIDATOS_UF)
    coluna_municipio = primeira_coluna_existente(colunas, CANDIDATOS_MUNICIPIO)

    filtros = []
    if coluna_uf:
        filtros.append(f"{coluna_uf} = 'RS'")
    if coluna_municipio:
        filtros.append(f"{coluna_municipio} = '{CODIGO_PORTO_ALEGRE_7}'")
    where = "WHERE " + " AND ".join(filtros) if filtros else ""

    if coluna_ano:
        sql = f"""
        SELECT
          MIN({coluna_ano}) AS ano_minimo,
          MAX({coluna_ano}) AS ano_maximo,
          COUNT(*) AS total_registros
        FROM `{tabela.tabela}`
        {where}
        """
    else:
        sql = f"""
        SELECT COUNT(*) AS total_registros
        FROM `{tabela.tabela}`
        {where}
        """
    return sql, coluna_ano, coluna_municipio


def montar_consulta_ano(tabela: TabelaCandidata, coluna_ano: str | None, colunas: set[str]) -> str | None:
    if coluna_ano is None:
        return None
    coluna_uf = primeira_coluna_existente(colunas, CANDIDATOS_UF)
    filtros = [f"{coluna_uf} = 'RS'"] if coluna_uf else []
    where = "WHERE " + " AND ".join(filtros) if filtros else ""
    return f"""
    SELECT
      {coluna_ano} AS ano,
      COUNT(*) AS total_registros
    FROM `{tabela.tabela}`
    {where}
    GROUP BY ano
    ORDER BY ano
    """


def formatar_gib(bytes_processados: int | None) -> str:
    if bytes_processados is None:
        return "nao estimado"
    return f"{bytes_processados / 1024**3:.3f} GiB"

## Auditoria das tabelas

Por padrao, o notebook executa apenas `dry_run` e consultas de metadados. Para executar as consultas de contagem, configure `EXECUTAR_CONSULTAS_BIGQUERY=1` no ambiente.

In [ ]:
client = obter_cliente_bigquery()
linhas_auditoria = []
series_por_ano = {}

for tabela in TABELAS:
    registro = {
        "fonte": "Base dos Dados",
        "tabela": tabela.tabela.replace("basedosdados.", ""),
        "uso_no_projeto": tabela.uso_previsto,
        "decisao": tabela.decisao_inicial,
        "periodo_disponivel": "a descobrir",
        "coluna_ano": None,
        "coluna_municipio": None,
        "bytes_estimados": "nao estimado",
        "status": "nao executado: configure GOOGLE_CLOUD_PROJECT e credenciais BigQuery para consultar",
    }

    if client is not None:
        try:
            colunas_df = executar_sql(client, consulta_colunas(tabela))
            colunas = set(colunas_df["column_name"].astype(str)) if not colunas_df.empty else set()
            sql_periodo, coluna_ano, coluna_municipio = montar_consulta_periodo(tabela, colunas)
            bytes_estimados = estimar_bytes(client, sql_periodo)
            registro.update(
                {
                    "coluna_ano": coluna_ano,
                    "coluna_municipio": coluna_municipio,
                    "bytes_estimados": formatar_gib(bytes_estimados),
                    "status": "dry_run ok",
                }
            )

            if EXECUTAR_CONSULTAS:
                periodo_df = executar_sql(client, sql_periodo)
                if not periodo_df.empty and "ano_minimo" in periodo_df.columns:
                    ano_minimo = periodo_df.loc[0, "ano_minimo"]
                    ano_maximo = periodo_df.loc[0, "ano_maximo"]
                    registro["periodo_disponivel"] = f"{ano_minimo}-{ano_maximo}"
                elif not periodo_df.empty:
                    registro["periodo_disponivel"] = "sem coluna anual identificada"

                sql_ano = montar_consulta_ano(tabela, coluna_ano, colunas)
                if sql_ano:
                    series_por_ano[tabela.tabela] = executar_sql(client, sql_ano)
                registro["status"] = "consulta executada"
        except Exception as exc:
            registro["status"] = f"erro: {type(exc).__name__}: {exc}"

    linhas_auditoria.append(registro)

matriz_auditoria = pd.DataFrame(linhas_auditoria)
matriz_auditoria

## Consultas-modelo

As consultas abaixo servem como referencia para auditoria manual no BigQuery quando o ambiente local nao tiver credenciais configuradas.

In [ ]:
consultas_modelo = {
    "SINASC RS": """
SELECT
  MIN(ano) AS ano_minimo,
  MAX(ano) AS ano_maximo,
  COUNT(*) AS total_registros
FROM `basedosdados.br_ms_sinasc.microdados`
WHERE sigla_uf = 'RS'
""",
    "SIM RS": """
SELECT
  ano,
  COUNT(*) AS total_registros
FROM `basedosdados.br_ms_sim.microdados`
WHERE sigla_uf = 'RS'
GROUP BY ano
ORDER BY ano
""",
    "CNES RS": """
SELECT
  ano,
  COUNT(*) AS total_registros
FROM `basedosdados.br_ms_cnes.estabelecimento`
WHERE sigla_uf = 'RS'
GROUP BY ano
ORDER BY ano
""",
    "Populacao municipal RS": """
SELECT
  ano,
  COUNT(*) AS total_registros
FROM `basedosdados.br_ms_populacao.municipio`
WHERE sigla_uf = 'RS'
GROUP BY ano
ORDER BY ano
""",
}

for nome, sql in consultas_modelo.items():
    print(f"--- {nome} ---")
    print(sql.strip())
    print()

## Visualizacao dos periodos auditados

A visualizacao e gerada apenas quando as consultas forem executadas e retornarem anos por tabela.

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

if plt is None:
    print("matplotlib nao instalado; visualizacao ignorada.")
elif not series_por_ano:
    print("Nenhuma serie por ano disponivel. Execute as consultas para gerar o grafico.")
else:
    periodos = []
    for tabela, df in series_por_ano.items():
        if not df.empty and "ano" in df.columns:
            periodos.append(
                {
                    "tabela": tabela.replace("basedosdados.", ""),
                    "ano_minimo": int(df["ano"].min()),
                    "ano_maximo": int(df["ano"].max()),
                }
            )

    periodos_df = pd.DataFrame(periodos).sort_values("tabela")
    fig, ax = plt.subplots(figsize=(10, 4.5))
    for idx, row in periodos_df.reset_index(drop=True).iterrows():
        ax.hlines(idx, row["ano_minimo"], row["ano_maximo"], linewidth=8, color="#0B7285")
        ax.text(row["ano_maximo"] + 0.1, idx, f"{row['ano_minimo']}-{row['ano_maximo']}", va="center")
    ax.set_yticks(range(len(periodos_df)))
    ax.set_yticklabels(periodos_df["tabela"])
    ax.set_xlabel("Ano disponível na Base dos Dados")
    ax.set_title("Cobertura temporal das tabelas auditadas")
    ax.grid(axis="x", alpha=0.25)
    fig.tight_layout()

    saida = Path("docs/assets/results/auditoria_basedosdados_periodos.png")
    if not saida.parent.exists():
        saida = Path("../../docs/assets/results/auditoria_basedosdados_periodos.png")
    saida.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(saida, dpi=200, bbox_inches="tight")
    print(f"Imagem salva em: {saida}")
    plt.show()

## Matriz de decisao

A matriz final deve orientar se cada tabela sera usada como validacao, fonte auxiliar, contexto ou apenas referencia tecnica.

In [ ]:
matriz_auditoria[["fonte", "tabela", "periodo_disponivel", "uso_no_projeto", "decisao", "bytes_estimados", "status"]]

In [ ]:
from src.visualization.generate_results import save_basedosdados_periods

audit_csv = Path("data/profiles/basedosdados_audit.csv")
if not audit_csv.exists():
    audit_csv = Path("../../data/profiles/basedosdados_audit.csv")

output = save_basedosdados_periods(audit_csv, Path("docs/assets/results"))
print(f"Imagem salva em: {output}")
